<a href="https://colab.research.google.com/github/vyenn/ML2024/blob/main/dnn_hw1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
!wget https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
!unzip data_gsn.zip &> /dev/null
!rm data_gsn.zip

--2025-11-03 02:32:21--  https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/marcin119a/data/refs/heads/main/data_gsn.zip [following]
--2025-11-03 02:32:22--  https://raw.githubusercontent.com/marcin119a/data/refs/heads/main/data_gsn.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5544261 (5.3M) [application/zip]
Saving to: ‘data_gsn.zip’

data_gsn.zip        100%[===================>]   5.29M  --.-KB/s    in 0.02s   

2025-11-03 02:32:23 (330 MB/s) - ‘data_gsn.zip’ saved [5544261/5544261]



In [2]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

In [3]:
class MyDataset(Dataset):
    def __init__(self, data_dir, csv_file, transform=None):
        self.data_dir = data_dir
        self.transform = transform

        # Read labels.csv
        self.labels_df = pd.read_csv(csv_file)

        # Assuming first column is filename, and next 6 columns are counts
        self.image_files = self.labels_df.iloc[:, 0].values  # file names
        self.cnt_labels = torch.tensor(self.labels_df.iloc[:, 1:7].values, dtype=torch.float)

        # Create the mapping from count_label to class_label | (6,) \mapsto (1,)
        self.count_to_class_id = {}
        class_id = 0
        for i in range(6):
            for j in range(i + 1, 6):
                for count_i in range(1, 10):
                    count_j = 10 - count_i

                    # Create the 6-D count vector
                    counts = [0] * 6
                    counts[i] = count_i
                    counts[j] = count_j

                    # Store the mapping (use a tuple as key)
                    self.count_to_class_id[tuple(counts)] = class_id
                    class_id += 1

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.data_dir, img_name)

        # Load and preprocess the image
        image = Image.open(img_path).convert("L")  # grayscale
        if self.transform:
            image = self.transform(image)

        # Load the count and class label
        cnt_label = self.cnt_labels[idx]
        cnt_tuple = tuple(cnt_label.int().tolist())
        cls_label = self.count_to_class_id[cnt_tuple]

        return image, cls_label, cnt_label,

## Loading data

In [4]:
data_dir = "data"
transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
csv_file = os.path.join(data_dir, "labels.csv")

dataset = MyDataset(data_dir=data_dir, csv_file=csv_file, transform=transform)
assert len(dataset) == 10000

# Split by index range
train_dataset = Subset(dataset, range(0, 9000))
val_dataset   = Subset(dataset, range(9000, 10000))


# small check
img, cls_lbl, cnt_lbl = train_dataset[0]
print(img.shape)
print(cls_lbl, cnt_lbl)

torch.Size([1, 28, 28])
93 tensor([0., 0., 4., 0., 6., 0.])


## Creating a model

it should return two outputs (log_probs, counts)

You may add dropout or batch normalization inside the heads, but you must not modify the backbone.



In [5]:
class ConvolutionalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=1, padding=1), nn.ReLU(),
            nn.Flatten(start_dim=1),
            nn.Linear(64 * 28 * 28, 256), nn.ReLU()
        )

        # Head 1: Classification
        # Takes the [B, 256] summary and outputs [B, 135] scores
        self.head_cls = nn.Linear(256, 135)
        """
        self.head_cls = nn.Sequential(
            nn.Linear(256, 128),       # An intermediate hidden layer
            nn.ReLU(),
            nn.Dropout(p=0.5),         # Dropout layer for regularization
            nn.Linear(128, 135)        # The final output layer
        )
        """

        # Head 2: Regression
        # Takes the [B, 256] summary and outputs [B, 6] count values
        self.head_cnt = nn.Linear(256, 6)
        """
        self.head_cnt = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),         # Add batch normalization
            nn.Linear(64, 6)
        )
        """

        # Helper to convert scores to log-probabilities, as requested
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        # x.shape = [B,1,28,28], features.shape = [B,256]
        features = self.feature_extractor(x)

        # Classification head
        cls_logits = self.head_cls(features)
        log_probs = self.log_softmax(cls_logits)

        # Regression head
        counts = self.head_cnt(features)

        return (log_probs, counts)

## Training prep

In [6]:
torch.manual_seed(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters from your requirements
LEARNING_RATE = 1e-3
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_VAL = 1000
N_EPOCHS = 100
TRAIN_SIZE = 9000
VAL_SIZE = 1000

# Early stopping parameters
PATIENCE = 10  # How many epochs to wait after last improvement
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_weights = None

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_VAL,
    shuffle=False # No need to shuffle validation data
)


# --- Model, Loss, and Optimizer ---
model = ConvolutionalNet().to(device)
# For head_cls (log-probabilities): Negative Log Likelihood Loss
criterion_cls = nn.NLLLoss()
# For head_cnt (regression): Mean Squared Error Loss (a common choice)
criterion_cnt = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Training

In [7]:
import copy

# --- The Training Loop ---
print("Starting training...")

for epoch in range(N_EPOCHS):

    model.train()
    running_train_loss = 0.0

    for inputs, labels_cls, labels_cnt in train_loader:
        # Move data to the correct device
        inputs = inputs.to(device)
        labels_cls = labels_cls.to(device)
        labels_cnt = labels_cnt.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        log_probs, counts = model(inputs)

        # Calculate losses
        loss_cls = criterion_cls(log_probs, labels_cls)
        # Ensure target is float for regression loss
        loss_cnt = criterion_cnt(counts, labels_cnt.float())

        # Total loss (simple sum)
        # ASSUMPTION: You can weigh these differently, e.g., loss = loss_cls + 0.5 * loss_cnt
        loss = loss_cls + loss_cnt

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_train_loss += loss.item() * inputs.size(0) # inputs.size(0) == BATCH_SIZE_TRAIN except from last sample since 9000%64=40

    # Calculate average training loss for the epoch
    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # --- Validation Phase ---
    model.eval()  # Set model to evaluation mode (disables dropout, etc.)
    running_val_loss = 0.0

    with torch.no_grad(): # Disable gradient calculations
        for inputs, labels_cls, labels_cnt in val_loader:
            # Move data to the correct device
            inputs = inputs.to(device)
            labels_cls = labels_cls.to(device)
            labels_cnt = labels_cnt.to(device)

            # Forward pass
            log_probs, counts = model(inputs)

            # Calculate losses
            loss_cls = criterion_cls(log_probs, labels_cls)
            loss_cnt = criterion_cnt(counts, labels_cnt.float())
            loss = loss_cls + loss_cnt

            running_val_loss += loss.item() * inputs.size(0)

    # Calculate average validation loss for the epoch
    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')

    # --- Early Stopping Check ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        # Save the weights of the best model
        best_model_weights = copy.deepcopy(model.state_dict())
        print('Validation loss improved. Saving model.')
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping triggered after {epoch+1} epochs.')
        break

print('Training finished.')

Starting training...
Epoch 1/100 | Train Loss: 9.5664 | Val Loss: 6.2578
Validation loss improved. Saving model.


KeyboardInterrupt: 